# Rainbow Six Siege: Data Mining & Preprocessing Pipeline
Η παρούσα ροή εργασίας (pipeline) υλοποιεί τον καθαρισμό, τον μετασχηματισμό και την εξαγωγή χαρακτηριστικών (feature engineering) από τα ακατέργαστα event logs του παιχνιδιού.

Η χρήση της βιβλιοθήκης `polars` επιλέχθηκε έναντι της `pandas` για τη βελτιστοποίηση της διαχείρισης μνήμης και του χρόνου εκτέλεσης, δεδομένου του μεγάλου όγκου των αρχείων (πολλαπλά GBs). Ο τελικός σκοπός είναι η παραγωγή δομημένων συνόλων δεδομένων μορφής `.arff` για την τροφοδότηση αλγορίθμων Μηχανικής Μάθησης (Classification, Clustering, Association Rules) στο περιβάλλον του Weka.


In [ ]:
import polars as pl
import glob, time, os

# ─── Περιβάλλον & Σταθερές ───────────────────────────────────────────
DATA_DIR = os.getcwd()
os.chdir(DATA_DIR)
print(f"📁 Working directory: {os.getcwd()}")

SEED = 2026
MIN_ROUND_DURATION, MAX_ROUND_DURATION, MIN_CLEARANCE_LEVEL = 45, 360, 20

# Ορισμός διαθέσιμων gadgets ανά ρόλο για sanity checks
ATTACKER_GADGETS = ["CLAYMORE", "FRAG_GRENADE", "SMOKE_GRENADE", "STUN_GRENADE", "BREACH_CHARGE"]
DEFENDER_GADGETS = ["BARBED_WIRE", "DEPLOYABLE_SHIELD", "NITRO_CELL", "IMPACT_GRENADE", "BULLETPROOF_CAMERA"]

# Ομαδοποίηση των Operators στους βασικούς τους ρόλους
OP_DATA = {
    "ASH": "Assault", "SLEDGE": "Assault", "CAPITAO": "Assault", "FUZE": "Assault", "STRIKER": "Assault",
    "BUCK": "Demolition", "BLITZ": "Shield", "MONTAGNE": "Shield", "BLACKBEARD": "Shield", "GLAZ": "Sniper", "IQ": "Recon", "JACKAL": "Recon",
    "THATCHER": "Recon", "TWITCH": "Recon", "THERMITE": "Support_ATK", "HIBANA": "Support_ATK",
    "JAGER": "Roamer", "BANDIT": "Roamer", "PULSE": "Roamer", "CAVEIRA": "Roamer",
    "SMOKE": "Anchor", "ECHO": "Anchor", "MIRA": "Anchor", "ROOK": "Anchor", "DOC": "Anchor", "SENTRY": "Anchor",
    "TACHANKA": "Anchor", "MUTE": "Trapper", "KAPKAN": "Trapper", "FROST": "Trapper",
    "VALKYRIE": "Intel", "CASTLE": "Intel", "RESERVE": "Anchor",
}


### Ενσωμάτωση Domain Knowledge (Χωρική & Τακτική Ανάλυση)

Η απλή ονομασία μιας τοποθεσίας (π.χ. 'LAUNDRY_ROOM') δεν φέρει εγγενή τακτική πληροφορία για τα μοντέλα μηχανικής μάθησης. Για την αντιμετώπιση αυτού του περιορισμού, εισάγεται το λεξικό `TSM_DATA`, το οποίο χαρτογραφεί κάθε τοποθεσία σε ένα διάνυσμα τριών διαστάσεων: `(Όροφος, Αριθμός Εξωτερικών Τοίχων, Αριθμός Καταπακτών)`. 

Η ποσοτικοποίηση αυτών των στοιχείων (π.χ. το υπόγειο στερείται εξωτερικών τοίχων αλλά έχει συχνά καταπακτές στην οροφή) επιτρέπει στους αλγόριθμους να εντοπίσουν μοτίβα που σχετίζονται με την κατακόρυφη άμυνα και τις απαιτήσεις προστασίας του εκάστοτε χώρου.


In [ ]:
TSM_DATA = {
    "OREGON": {
        "LAUNDRY_ROOM": (-1, 0, 2), "LAUDRY_ROOM-SUPPLY_ROOM": (-1, 0, 2), "SUPPLY": (-1, 0, 2),
        "KITCHEN": (0, 1, 1), "KITCHEN-DINING_HALL": (0, 1, 1), "DINING_HALL": (0, 1, 1),
        "KIDS_DORMS-DORMS_MAIN_HALL": (1, 2, 0), "DORMS_MAIN_HALL": (1, 2, 0),
        "MEETING_HALL": (0, 1, 1), "REAR_STAGE-WATCH_TOWER": (0, 1, 1)
    },
    "CONSULATE": {
        "GARAGE": (-1, 1, 1), "GARAGE-CAFETERIA": (-1, 1, 1), "ARCHIVES": (-1, 0, 2),
        "CEO_OFFICE": (1, 3, 0), "CONSUL_OFFICE-MEEETING_ROOM": (1, 3, 0), "PRESS_ROOM": (0, 2, 1),
        "TELLERS": (0, 2, 1), "VISA_OFFICE": (0, 2, 1), "ADMINISTRATION_OFFICE": (1, 3, 0)
    },
    "CLUB_HOUSE": {
        "CHURCH-ARSENAL_ROOM": (-1, 0, 2), "CHURCH": (-1, 0, 2), "ARSENAL_ROOM": (-1, 0, 3),
        "GYM-BEDROOM": (1, 2, 0), "BEDROOM": (1, 2, 0), "CASH_ROOM": (1, 2, 1),
        "CCTV_ROOM-CASH_ROOM": (1, 2, 1), "BAR-STOCK_ROOM": (0, 2, 1), "STRIP_CLUB": (0, 2, 1)
    },
    "BANK": {
        "LOCKERS-CCTV_ROOM": (-1, 0, 3), "LOCKERS": (-1, 0, 3), "VAULT": (-1, 0, 1),
        "TELLERS_OFFICE-ARCHIVES": (0, 2, 1), "TELLER'S_OFFICE": (0, 2, 1), "STAFF_ROOM-OPEN_AREA": (0, 2, 1),
        "CEO_OFFICE": (1, 3, 0), "EXECUTIVE_LOUNGE-CEO_OFFICE": (1, 3, 0)
    },
    "BORDER": {
        "ARMORY_LOCKERS": (1, 3, 0), "ARMORY_LOCKERS-ARCHIVES": (1, 3, 0),
        "VENTILATION_ROOM-WORKSHOP": (0, 1, 1), "WORKSHOP": (0, 2, 1), "TELLERS": (0, 1, 1),
        "CUSTOMS_INSPECTIONS-SUPPLY_ROOM": (0, 2, 1), "BATHROOM-TELLERS": (0, 1, 1)
    },
    "CHALET": {
        "WINE_CELLAR-SNOWMOBILE_GARAGE": (-1, 1, 1), "SNOWMOBILE_GARAGE": (-1, 1, 1),
        "KITCHEN-TROPHY_ROOM": (0, 2, 1), "KITCHEN": (0, 2, 1), "BAR-GAMING_ROOM": (0, 2, 1),
        "MASTER_BEDROOM-OFFICE": (1, 2, 0), "MASTER_BEDROOM": (1, 2, 0)
    },
    "HOUSE": {
        "KID'S_BEDROOM-WORKSHOP": (1, 3, 0), "KID'S_BEDROOM": (1, 3, 0), "MASTER_BEDROOM": (1, 3, 0),
        "LIVING_ROOM-TRAINING_ROOM": (0, 3, 1), "LIVING_ROOM": (0, 3, 1), "GARAGE": (-1, 2, 1),
        "TRAINING_ROOM-GARAGE": (-1, 2, 1), "LAUNDRY_ROOM": (-1, 2, 1)
    },
    "PLANE": {
        "MEETING_ROOM": (1, 0, 2), "EXECUTIVE_BEDROOM": (1, 0, 2), "STAFF_SECTION-EXECUTIVE_BEDROOM": (1, 0, 2),
        "CARGO_HOLD-LUGGAGE_HOLD": (-1, 0, 3), "LUGGAGE_HOLD": (-1, 0, 3)
    },
    "COASTLINE": {
        "2F_PENTHOUSE-2F_THEATER": (1, 2, 1), "2F_HOOKAH_LOUNGE-2F_BILLIARDS_ROOM": (1, 3, 0),
        "1F_KITCHEN-1F_SERVICE_ENTRANCE": (0, 2, 1), "1F_BLUE_BAR-1F_SUNRISE_BAR": (0, 3, 1)
    },
    "KAFE_DOSTOYEVSKY": {
        "BAR-COCKTAIL_LOUNGE": (2, 2, 1), "CIGAR_LOUNGE": (2, 2, 1), "READING_ROOM": (1, 2, 1),
        "FIREPLACE_HALL-MINING_ROOM": (1, 2, 1), "KITCHEN_PREP-BAKERY": (0, 2, 1)
    },
    "SKYSCRAPER": {
        "2F_KARAOKE-2F_TEA_ROOM": (1, 2, 0), "2F_TEA_ROOM": (1, 2, 0), "2F_EXHIBITION-2F_WORK_OFFICE": (1, 3, 0),
        "1F_BBQ-1F_KITCHEN": (0, 2, 1), "1F_BEDROOM-1F_BATHROOM": (0, 2, 1)
    },
    "KANAL": {
        "SERVER_ROOM-CONTROL_ROOM": (2, 3, 0), "CONTROL_ROOM": (2, 3, 0), "MAPS_OFFICE": (1, 3, 0),
        "COAST_GUARD_OFFICE-HOLDING_ROOM": (0, 3, 1), "BOAT_SUPPLIES": (-1, 0, 2)
    },
    "YACHT": {
        "COCKPIT": (2, 3, 0), "MAPS_ROOM-COCKPIT": (2, 3, 0), "CASINO": (1, 3, 0),
        "KITCHEN-ENGINE_CONTROL": (0, 2, 1), "SERVER_ROOM-ENGINE_STORAGE": (-1, 0, 2)
    },
    "FAVELAS": {
        "3F_PACKAGING_ROOM-2F_METH_LAB": (2, 4, 0), "2F_AUNT'S_BEDROOM-1F_AUNT'S_APARTMENT": (1, 3, 1),
        "2F_FOOTBALL_BEDROOM-2F_FOOTBALL_OFFICE": (1, 3, 1), "1F_BIKER'S_APARTMENT-1F_BIKER'S_BEDROOM": (0, 2, 1)
    },
    "BARTLETT_U.": {
        "READING_ROOM-LIBRARY": (1, 2, 1), "ROWING_MUSEUM-TROPHY_ROOM": (0, 2, 1),
        "KITCHEN-PIANO_ROOM": (0, 2, 1), "CLASSROOM-LIBRARY": (0, 2, 1)
    },
    "HEREFORD_BASE": {
        "ARMORY": (-1, 0, 3), "BRIEFING_ROOM-ARMORY": (-1, 0, 3), "KITCHEN": (0, 2, 1),
        "TV_ROOM-KITCHEN": (0, 2, 1), "DINING_ROOM-KIDS_BEDROOM": (1, 2, 0)
    }
}

WPN_DATA = {
    # Καραμπίνες
    "M590A1": "Shotgun", "M1014": "Shotgun", "SG-CQB": "Shotgun", "SUPER_90": "Shotgun", "SASG-12": "Shotgun", "SPAS-12": "Shotgun", "SPAS-15": "Shotgun", "ITA12L": "Shotgun", "ITA12S": "Shotgun", "SUPERNOVA": "Shotgun", "SuperNova": "Shotgun", "M870": "Shotgun",
    # Τουφέκια Ακριβείας (DMR)
    "417": "DMR", "HK417": "DMR", "CAMRS": "DMR", "SR-25": "DMR", "OTS-03": "DMR", "OTs-03": "DMR",
    # Ασπίδες
    "Shield": "Shield", "BALLISTIC_SHIELD": "Shield", "LE_ROC": "Shield", "EXTENDABLE_SHIELD": "Shield", "BLITZ_SHIELD": "Shield",
}

df_tactical = pl.DataFrame([{"mapname": m, "objectivelocation": s, "floor_level": v[0], "external_soft_walls": v[1], "crucial_hatches": v[2]} for m, sites in TSM_DATA.items() for s, v in sites.items()])
df_wpns = pl.DataFrame([{"primaryweapon": k, "Weapon_Category": v} for k, v in WPN_DATA.items()])
df_ops = pl.DataFrame([{"operator": k, "Operator_Role": v} for k, v in OP_DATA.items()])

COLUMNS_TO_KEEP = ["matchid", "team", "roundnumber", "gamemode", "mapname", "objectivelocation", "skillrank", "role", "haswon", "operator", "nbkills", "isdead", "roundduration", "clearancelevel", "endroundreason", "primaryweapon", "secondarygadget"]
COLUMN_RENAME_MAP = {
    "mapname": "Map_Name", "objectivelocation": "Objective_Location", "skillrank": "Skill_Rank", 
    "role": "Role", "haswon": "Round_Result_Won", "operator": "Operator_Name", 
    "nbkills": "Total_Kills", "isdead": "Player_Died", "roundduration": "Round_Duration_Seconds", 
    "clearancelevel": "Clearance_Level", "endroundreason": "End_Round_Reason", 
    "primaryweapon": "Primary_Weapon", "secondarygadget": "Secondary_Gadget"
}
LETHAL_WEIGHTS = {"FRAG_GRENADE": 3, "NITRO_CELL": 3, "CLAYMORE": 2, "IMPACT_GRENADE": 1}
TACTICAL_WEIGHTS = {"SMOKE_GRENADE": 3, "DEPLOYABLE_SHIELD": 3, "BULLETPROOF_CAMERA": 2, "STUN_GRENADE": 1, "BREACH_CHARGE": 1, "BARBED_WIRE": 1}


### Προεπεξεργασία & Καθαρισμός Δεδομένων (Data Cleaning)

Η συνάρτηση `build_clean_pipeline` εφαρμόζει τα αρχικά φίλτρα ακεραιότητας στα δεδομένα:
- Περιορισμός των δεδομένων αποκλειστικά στο `BOMB` gamemode, το οποίο αποτελεί το πρότυπο του ανταγωνιστικού παιχνιδιού.
- Απόρριψη γύρων με διάρκεια μικρότερη των 45 δευτερολέπτων (ενδεικτικό τεχνικών σφαλμάτων ή αποσυνδέσεων).
- Αφαίρεση ακραίων τιμών (outliers), όπως καταγραφές παικτών με περισσότερα από 5 kills ανά γύρο.
- Εξαγωγή μετα-χαρακτηριστικών όπως το `Match_Phase` (χρονική φάση του αγώνα) και το `Win_Method`.


In [ ]:
def build_clean_pipeline(lazy: pl.LazyFrame) -> pl.LazyFrame:
    return (
        lazy.select(COLUMNS_TO_KEEP).drop_nulls()
        .filter((pl.col("roundduration") >= MIN_ROUND_DURATION) & (pl.col("roundduration") <= MAX_ROUND_DURATION))
        .filter(pl.col("clearancelevel") >= MIN_CLEARANCE_LEVEL)
        .filter(pl.col("gamemode") == "BOMB")
        .filter(pl.col("nbkills") <= 5).filter(pl.col("roundnumber") <= 9)
        .with_columns([
            pl.col("mapname").str.replace_all("'", "").str.to_uppercase(),
            pl.col("objectivelocation").str.replace_all("'", "").str.to_uppercase(),
            pl.col("operator").str.replace_all("'", "").str.to_uppercase(),
        ])
        .with_columns(pl.when(pl.col("operator").str.contains("(?i)Recruit") | (pl.col("operator") == "RESERVE")).then(pl.when(pl.col("role") == "Attacker").then(pl.lit("STRIKER")).otherwise(pl.lit("SENTRY"))).otherwise(pl.col("operator").str.replace(r"^.*-", "")).alias("operator"))
        .with_columns(pl.col("operator").replace_strict(OP_DATA, default="Other").alias("Operator_Role"))
        .filter(~((pl.col("role") == "Defender") & ~pl.col("secondarygadget").is_in(DEFENDER_GADGETS)) & ~((pl.col("role") == "Attacker") & ~pl.col("secondarygadget").is_in(ATTACKER_GADGETS)))
        .with_columns([
            pl.when(pl.col("endroundreason").is_in(["DefuserDeactivated", "ObjectiveCaptured", "HostageExtracted"])).then(pl.lit("Objective_Play")).when(pl.col("endroundreason").is_in(["AttackersEliminated", "DefendersEliminated"])).then(pl.lit("Deathmatch")).when(pl.col("endroundreason") == "TimeExpired").then(pl.lit("Time_Stall")).otherwise(pl.lit("Other")).alias("Win_Method"),
            pl.when(pl.col("roundnumber") <= 2).then(pl.lit("Early_Game")).when(pl.col("roundnumber") <= 4).then(pl.lit("Mid_Game")).otherwise(pl.lit("Late_Game")).alias("Match_Phase"),
            pl.when((pl.col("nbkills") >= 2) & (pl.col("roundduration") < 120)).then(pl.lit("Aggressive_Rush")).when((pl.col("nbkills") <= 1) & (pl.col("roundduration") >= 180)).then(pl.lit("Tactical_Passive")).otherwise(pl.lit("Standard")).alias("Playstyle")
        ])
    )


### Έλεγχος Πληρότητας (5v5 Integrity Check) & Δειγματοληψία

Για τη διασφάλιση της ορθότητας των μοντέλων μηχανικής μάθησης, είναι κρίσιμο να αναλύονται μόνο γύροι όπου και οι δύο ομάδες ήταν πλήρεις (5 εναντίον 5). Η παρακάτω διαδικασία σαρώνει τα δεδομένα και απορρίπτει γύρους όπου σημειώθηκαν αποσυνδέσεις παικτών ή αποκλεισμοί.

Λόγω του τεράστιου όγκου των παραγόμενων γύρων, εφαρμόζεται στρωματοποιημένη δειγματοληψία (stratified sampling) 40.000 γύρων. Το μέγεθος αυτό κρίνεται επαρκές για την εξαγωγή στατιστικά σημαντικών συμπερασμάτων, διατηρώντας ταυτόχρονα το υπολογιστικό κόστος σε βιώσιμα επίπεδα για το Weka.


In [ ]:
def run_master_pipelines(sample_n_rounds=40000):
    print(f"Έναρξη επεξεργασίας (Στόχος: {sample_n_rounds:,} γύροι)...")
    all_files = sorted(glob.glob("datadump_s5-*.csv"))

    # Διαβάζουμε όλα τα αρχεία μόνο μία φορά για εξοικονόμηση χρόνου
    # Εφαρμόζουμε αμέσως τα φίλτρα ποιότητας
    # Κρατάμε μόνο τους γύρους που έμειναν 5 εναντίον 5 μετά τα φίλτρα
    # Αυτό εξασφαλίζει την ακεραιότητα των δεδομένων
    print("Σάρωση αρχείων — Εφαρμογή φίλτρων και έλεγχος για 5v5...")
    all_frames = []
    for i, filepath in enumerate(all_files):
        df = build_clean_pipeline(pl.scan_csv(filepath, ignore_errors=True)).collect()
        if df.is_empty():
            continue
            
        # Αποφυγή διπλότυπων ID αγώνα προσθέτοντας το όνομα του χάρτη
        df = df.with_columns(
            (pl.col("matchid").cast(pl.String) + "_" + pl.col("mapname") + "_" + str(i)).alias("matchid")
        )

        # Έλεγχος για 5 Επιτιθέμενους και 5 Αμυνόμενους
        complete = (
            df.group_by(["matchid", "roundnumber"])
            .agg([
                pl.col("role").filter(pl.col("role") == "Attacker").len().alias("atk_n"),
                pl.col("role").filter(pl.col("role") == "Defender").len().alias("def_n"),
            ])
            .filter((pl.col("atk_n") == 5) & (pl.col("def_n") == 5))
            .select(["matchid", "roundnumber"])
        )
        df = df.join(complete, on=["matchid", "roundnumber"])
        if not df.is_empty():
            all_frames.append(df)

    # Συγκέντρωση όλων των έγκυρων γύρων από όλα τα αρχεία
    pool = pl.concat(all_frames).unique(subset=["matchid", "roundnumber", "team", "operator"])
    valid_ids = pool.select(["matchid", "roundnumber"]).unique()
    print(f"Βρέθηκαν {valid_ids.height:,} συνολικοί έγκυροι γύροι μετά τα φίλτρα.")

    # Δημιουργία τυχαίου δείγματος
    sampled_ids = valid_ids.sample(n=min(sample_n_rounds, valid_ids.height), seed=SEED)
    print(f"Λήψη δείγματος {sampled_ids.height:,} γύρων για ανάλυση.")

    df_full_pool = pool.join(sampled_ids, on=["matchid", "roundnumber"])
    n_rounds = df_full_pool.select(["matchid", "roundnumber"]).unique().height
    print(f"Υπολογισμός νέων χαρακτηριστικών για {n_rounds:,} γύρους...")


### Feature Engineering (Μοντελοποίηση Τακτικής & Momentum)

Σε αυτό το στάδιο εξάγονται τα σύνθετα χαρακτηριστικά (features) που περιγράφουν τη δυναμική της αναμέτρησης:
- **Εμπειρία Ομάδας:** Υπολογισμός του μέσου Clearance Level ανά ομάδα.
- **Ψυχολογικό Momentum:** Εξαγωγή σερί νικών (win streaks) και ιστορικού απόδοσης (Kills/Deaths προηγούμενων γύρων).
- **Τακτική Σύνθεση:** Καταμέτρηση των ρόλων (π.χ. Anchors, Roamers) και του οπλισμού (π.χ. Shotguns, Shields) ανά ομάδα.

Το αποτέλεσμα είναι η δημιουργία τριών διακριτών συνόλων δεδομένων, το καθένα προσαρμοσμένο για διαφορετικούς αλγορίθμους (Apriori σε επίπεδο Operators, Apriori σε επίπεδο Ρόλων, και ένα ενοποιημένο σύνολο για Classification).


In [ ]:
# Υπολογισμός μέσου όρου εμπειρίας (Clearance Level) ανά ομάδα
    df_team_exp = (df_full_pool.group_by(["matchid", "roundnumber", "role"])
                   .agg(pl.col("clearancelevel").mean().alias("avg_clearance"))
                   .with_columns(
                       pl.when(pl.col("avg_clearance") < 100).then(pl.lit("Novice (<100)"))
                       .when(pl.col("avg_clearance") <= 150).then(pl.lit("Veteran (100-150)"))
                       .otherwise(pl.lit("Elite (>150)"))
                       .alias("exp_label")
                   ))
    
    # Τα χαρακτηριστικά DK (Domain Knowledge) δημιουργήθηκαν από εμάς
    atk_exp = df_team_exp.filter(pl.col("role") == "Attacker").select(["matchid", "roundnumber", "exp_label"]).rename({"exp_label": "DK_ATK_Experience_Level"})
    def_exp = df_team_exp.filter(pl.col("role") == "Defender").select(["matchid", "roundnumber", "exp_label"]).rename({"exp_label": "DK_DEF_Experience_Level"})

    # Υπολογισμός πληροφοριών γύρου (Χάρτης, Τοποθεσία, Φάση)
    # Διαγραφή της αρχικής αριθμητικής στήλης ορόφου
    # Διόρθωση ασυμφωνίας μεταξύ κενών τιμών
    df_apriori_context = (
        df_full_pool.group_by(["matchid", "roundnumber"]).agg([
            pl.col("mapname").first().alias("Map_Name"),
            pl.col("objectivelocation").first().alias("Objective_Location"),
            pl.col("roundnumber").first().alias("raw_rn")
        ]).with_columns([
            pl.when(pl.col("raw_rn") <= 2).then(pl.lit("Early_Game"))
            .when(pl.col("raw_rn") <= 4).then(pl.lit("Mid_Game"))
            .otherwise(pl.lit("Late_Game")).alias("DK_Match_Phase")
        ]).join(
            df_tactical.select(["mapname", "objectivelocation", "floor_level"]),
            left_on=["Map_Name", "Objective_Location"],
            right_on=["mapname", "objectivelocation"],
            how="left"
        ).with_columns(
            # Γέμισμα των κενών με 0 (ισόγειο) για να μην υπάρχουν ελλιπή δεδομένα
            pl.col("floor_level").fill_null(0).cast(pl.String).alias("DK_Floor_Level")
        ).drop("floor_level")
        .join(atk_exp, on=["matchid", "roundnumber"])
        .join(def_exp, on=["matchid", "roundnumber"])
    )
    
    # Διαχωρισμός και ομαδοποίηση των Επιτιθέμενων
    # Αφαίρεση διπλότυπων operators για να μην χτυπήσει σφάλμα
    # Εξαγωγή του αποτελέσματος νίκης. Πετάμε γύρους με αντικρουόμενα δεδομένα
    # (λάθη καταγραφής στο ίδιο το παιχνίδι)
    df_haswon_lookup = (
        df_full_pool
        .filter(pl.col("role") == "Attacker")
        .group_by(["matchid", "roundnumber"])
        .agg([
            pl.col("haswon").min().alias("hw_min"),
            pl.col("haswon").max().alias("hw_max"),
        ])
        .filter(pl.col("hw_min") == pl.col("hw_max"))   # Κρατάμε μόνο γύρους που όλοι έχουν το ίδιο αποτέλεσμα
        .rename({"hw_min": "haswon"})
        .drop("hw_max")
    )
    df_atk = (df_full_pool
              .filter(pl.col("role") == "Attacker")
              .select(["matchid", "roundnumber", "operator"])
              .unique(subset=["matchid", "roundnumber", "operator"], keep="first"))
    df_atk_pivot = (df_atk.with_columns(pl.lit(True).alias("present"))
                    .pivot(index=["matchid", "roundnumber"], on="operator", values="present")
                    .join(df_haswon_lookup, on=["matchid", "roundnumber"]))  # Πετάμε τα χαλασμένα δεδομένα
    df_atk_pivot = df_atk_pivot.rename({c: f"ATK_{c}" for c in df_atk_pivot.columns if c not in ["matchid", "roundnumber", "haswon"]})


    # Διαχωρισμός και ομαδοποίηση των Αμυνόμενων
    # Αφαίρεση διπλότυπων και για τους Αμυνόμενους
    df_def = (df_full_pool
              .filter(pl.col("role") == "Defender")
              .select(["matchid", "roundnumber", "operator"])
              .unique(subset=["matchid", "roundnumber", "operator"], keep="first"))
    df_def_pivot = (df_def.with_columns(pl.lit(True).alias("present"))
                    .pivot(index=["matchid", "roundnumber"], on="operator", values="present"))
    df_def_pivot = df_def_pivot.rename({c: f"DEF_{c}" for c in df_def_pivot.columns if c not in ["matchid", "roundnumber"]})
    
    # Ένωση σε έναν πίνακα με όλο το ματς (Matchup)
    df_apriori = df_atk_pivot.join(df_def_pivot, on=["matchid", "roundnumber"])
    
    # Προσθήκη των τακτικών πληροφοριών (Context)
    df_apriori = df_apriori.join(df_apriori_context.drop(["raw_rn"]), on=["matchid", "roundnumber"])
    
    # Αφαίρεση τεχνικών ID που δεν χρειάζονται πια
    df_apriori = df_apriori.drop(["matchid", "roundnumber"])
    
    # Μετονομασία της νίκης και μετατροπή σε Κείμενο (String)
    # για να το αναγνωρίσει σωστά το Weka ως κατηγορία
    df_apriori = df_apriori.rename({"haswon": "Attacker_Won"})
    df_apriori = df_apriori.with_columns(
        pl.col("Attacker_Won").cast(pl.String)  # Εξασφαλίζει ότι θα εξαχθεί ως κατηγορία, όχι ως αριθμός
    )
    # Μετακίνηση του Target Attribute (Νίκη) στην τελευταία στήλη
    other_cols = [c for c in df_apriori.columns if c != "Attacker_Won"]
    df_apriori = df_apriori.select(other_cols + ["Attacker_Won"])

    # ─── Dataset για Apriori με βάση τους Ρόλους ─────────────────────────
    # Αντί για μεμονωμένους operators, μετράμε τους ρόλους
    # Η μέτρηση γίνεται κείμενο για το Weka
    df_ops_roles = pl.DataFrame([{"operator": k, "Operator_Role": v} for k, v in OP_DATA.items()])

    df_roles_base = (
        df_full_pool
        .select(["matchid", "roundnumber", "role", "haswon", "operator"])
        .unique(subset=["matchid", "roundnumber", "operator"], keep="first")
        .join(df_ops_roles, on="operator", how="left")
        .with_columns(pl.col("Operator_Role").fill_null("Unknown"))
    )

    # Αριθμός ρόλων Επιτιθέμενων (τα μηδενικά γίνονται κενά για να αγνοηθούν)
    df_atk_roles = (
        df_roles_base.filter(pl.col("role") == "Attacker")
        .select(["matchid", "roundnumber", "Operator_Role"])
        .group_by(["matchid", "roundnumber", "Operator_Role"])
        .agg(pl.len().alias("cnt"))
        .pivot(index=["matchid", "roundnumber"], on="Operator_Role", values="cnt")
        # Διατήρηση null τιμών για την ορθή επεξεργασία από τον αλγόριθμο Apriori
        .join(df_haswon_lookup, on=["matchid", "roundnumber"])
    )
    df_atk_roles = df_atk_roles.rename({
        c: f"ATK_{c}" for c in df_atk_roles.columns
        if c not in ["matchid", "roundnumber", "haswon"]
    })
    role_cols_atk = [c for c in df_atk_roles.columns if c.startswith("ATK_")]
    df_atk_roles = df_atk_roles.with_columns(
        [pl.col(c).cast(pl.Int32).cast(pl.String) for c in role_cols_atk]
        # Το κενό παραμένει null για την αναγνώριση απουσίας αντικειμένου
    )

    # Αριθμός ρόλων Αμυνόμενων (ίδια λογική με τα κενά)
    df_def_roles = (
        df_roles_base.filter(pl.col("role") == "Defender")
        .select(["matchid", "roundnumber", "Operator_Role"])
        .group_by(["matchid", "roundnumber", "Operator_Role"])
        .agg(pl.len().alias("cnt"))
        .pivot(index=["matchid", "roundnumber"], on="Operator_Role", values="cnt")
        # Το 0 γίνεται κενό
    )
    df_def_roles = df_def_roles.rename({
        c: f"DEF_{c}" for c in df_def_roles.columns
        if c not in ["matchid", "roundnumber"]
    })
    role_cols_def = [c for c in df_def_roles.columns if c.startswith("DEF_")]
    df_def_roles = df_def_roles.with_columns(
        [pl.col(c).cast(pl.Int32).cast(pl.String) for c in role_cols_def]
    )

    # Ένωση των ρόλων με τις τακτικές πληροφορίες
    df_apriori_roles = (
        df_atk_roles
        .join(df_def_roles, on=["matchid", "roundnumber"])
        # Χρησιμοποιούμε μόνο τον Χάρτη, τον Όροφο και την Φάση του παιχνιδιού
        # Η ακριβής τοποθεσία αφαιρέθηκε γιατί έχει πολλές διαφορετικές τιμές
        # Η εμπειρία αφαιρέθηκε για να μην μπερδέψει τα τακτικά μοτίβα
        .join(
            df_apriori_context.select(["matchid", "roundnumber", "Map_Name", "DK_Floor_Level", "DK_Match_Phase"]),
            on=["matchid", "roundnumber"]
        )
        .drop(["matchid", "roundnumber"])
        .rename({"haswon": "Attacker_Won"})
        .with_columns(pl.col("Attacker_Won").cast(pl.String))
    )
    other_cols_r = [c for c in df_apriori_roles.columns if c != "Attacker_Won"]
    df_apriori_roles = df_apriori_roles.select(other_cols_r + ["Attacker_Won"])

    
    # 2. Δημιουργία Γενικού Δείγματος για Ταξινόμηση (Classification)
    print("Υπολογισμός στρατηγικών χαρακτηριστικών για το Γενικό Δείγμα...")
    df_full_pool = df_full_pool.join(df_wpns, on="primaryweapon", how="left")
    df_full_pool = df_full_pool.with_columns([
        pl.col("Weapon_Category").fill_null("Auto"),
        pl.col("secondarygadget").replace_strict(LETHAL_WEIGHTS, default=0).cast(pl.Int32).alias("Lethal_Contrib"),
        pl.col("secondarygadget").replace_strict(TACTICAL_WEIGHTS, default=0).cast(pl.Int32).alias("Tactical_Contrib"),
        pl.col("haswon").cast(pl.Int32).alias("haswon_numeric"),
        pl.col("nbkills").cast(pl.Int32),
        pl.col("isdead").cast(pl.Int32),
        pl.col("clearancelevel").cast(pl.Float32)
    ])
    
    team_timeline = df_full_pool.group_by(["matchid", "team", "roundnumber"]).agg([
        pl.col("haswon_numeric").first().alias("won_round"),
        pl.col("nbkills").sum().alias("team_kills"),
        pl.col("isdead").sum().alias("team_deaths"),
        pl.col("clearancelevel").mean().alias("team_avg_clearance"),
        pl.col("Lethal_Contrib").sum().alias("Team_Lethal_Score"),
        pl.col("Tactical_Contrib").sum().alias("Team_Tactical_Score"),
        (pl.col("nbkills") == 5).any().cast(pl.Int32).alias("got_ace"),
        (pl.col("Weapon_Category") == "Shotgun").cast(pl.Int32).sum().alias("team_shotgun_count"),
        (pl.col("Weapon_Category") == "DMR").cast(pl.Int32).sum().alias("team_dmr_count"),
        (pl.col("Weapon_Category") == "Shield").cast(pl.Int32).sum().alias("team_shield_count"),
        (pl.col("Weapon_Category") == "Auto").cast(pl.Int32).sum().alias("team_auto_count")
    ]).sort(["matchid", "team", "roundnumber"]).with_columns([
        pl.col("won_round").shift(1).over(["matchid", "team"]).fill_null(-1).alias("Prev_Round_Won"),
        pl.col("won_round").shift(1).over(["matchid", "team"]).fill_null(0).cum_sum().over(["matchid", "team"]).alias("My_Team_Score"),
        pl.col("team_kills").shift(1).over(["matchid", "team"]).fill_null(0).alias("raw_prev_kills"),
        pl.col("team_deaths").shift(1).over(["matchid", "team"]).fill_null(0).alias("raw_prev_deaths"),
        pl.col("team_kills").shift(1).over(["matchid", "team"]).fill_null(0).cum_sum().over(["matchid", "team"]).alias("match_total_kills"),
        pl.col("team_deaths").shift(1).over(["matchid", "team"]).fill_null(0).cum_sum().over(["matchid", "team"]).alias("match_total_deaths")
    ]).with_columns([
        pl.when(pl.col("raw_prev_kills") == 0).then(pl.lit("None")).when(pl.col("raw_prev_kills") <= 2).then(pl.lit("Low (1-2)")).when(pl.col("raw_prev_kills") <= 4).then(pl.lit("Mid (3-4)")).otherwise(pl.lit("High (5)")).alias("Prev_Round_Kills_Bucket"),
        pl.when(pl.col("raw_prev_deaths") == 0).then(pl.lit("None (Flawless)")).when(pl.col("raw_prev_deaths") <= 2).then(pl.lit("Low (1-2)")).when(pl.col("raw_prev_deaths") <= 4).then(pl.lit("High (3-4)")).otherwise(pl.lit("Wiped (5)")).alias("Prev_Round_Deaths_Bucket"),
        pl.when(pl.col("match_total_kills") <= 3).then(pl.lit("Early_Game/Low")).when(pl.col("match_total_kills") <= 8).then(pl.lit("8_Kills")).when(pl.col("match_total_kills") <= 12).then(pl.lit("12_Kills")).when(pl.col("match_total_kills") <= 15).then(pl.lit("15_Kills")).otherwise(pl.lit("Elite (>15)")).alias("Match_Kills_Bucket"),
        pl.when(pl.col("match_total_deaths") <= 3).then(pl.lit("Low_Deaths")).when(pl.col("match_total_deaths") <= 8).then(pl.lit("8_Deaths")).when(pl.col("match_total_deaths") <= 12).then(pl.lit("12_Deaths")).when(pl.col("match_total_deaths") <= 15).then(pl.lit("15_Deaths")).otherwise(pl.lit("High_Losses (>15)")).alias("Match_Deaths_Bucket"),
        pl.when(pl.col("team_avg_clearance") < 100).then(pl.lit("Novice (<100)")).when(pl.col("team_avg_clearance") <= 150).then(pl.lit("Veteran (100-150)")).otherwise(pl.lit("Elite (>150)")).alias("Team_Experience_Level")
    ])
    
    team_timeline = team_timeline.with_columns(
        (pl.col("won_round") != pl.col("won_round").shift(1).over(["matchid", "team"])).fill_null(True).cum_sum().over(["matchid", "team"]).alias("streak_group")
    ).with_columns(
        pl.col("won_round").cum_count().over(["matchid", "team", "streak_group"]).alias("current_streak_raw")
    ).with_columns(
        pl.when(pl.col("won_round").shift(1).over(["matchid", "team"]) == 1).then(pl.col("current_streak_raw").shift(1).over(["matchid", "team"])).otherwise(pl.col("current_streak_raw").shift(1).over(["matchid", "team"]) * -1).fill_null(0).alias("Win_Streak")
    )
    
    opponent_scores = (team_timeline.select(["matchid", "roundnumber", "team", "My_Team_Score"]).join(team_timeline.select(["matchid", "roundnumber", "team", "My_Team_Score"]), on=["matchid", "roundnumber"], suffix="_opp").filter(pl.col("team") != pl.col("team_opp")).select(["matchid", "roundnumber", "team", "My_Team_Score_opp"]).rename({"My_Team_Score_opp": "Opponent_Score"}))
    df_full_enriched = (df_full_pool.join(team_timeline, on=["matchid", "team", "roundnumber"]).join(opponent_scores, on=["matchid", "team", "roundnumber"]))
    
    print("👑 Προσθήκη τακτικών πληροφοριών χάρτη...")
    df_full_enriched = df_full_enriched.join(df_tactical, on=["mapname", "objectivelocation"], how="left")
    df_full_enriched = df_full_enriched.with_columns([pl.col(c).fill_null(0).cast(pl.Int32) for c in ["floor_level", "external_soft_walls", "crucial_hatches"]])

    team_role_stats = df_full_enriched.group_by(["matchid", "roundnumber", "team"]).agg([pl.col("clearancelevel").mean().alias("team_avg_clearance_numeric"), pl.col("Operator_Role").n_unique().alias("team_diversity_score"), (pl.col("Operator_Role") == "Anchor").cast(pl.Int32).sum().alias("count_Anchor"), (pl.col("Operator_Role") == "Assault").cast(pl.Int32).sum().alias("count_Assault"), (pl.col("Operator_Role") == "Roamer").cast(pl.Int32).sum().alias("count_Roamer"), (pl.col("Operator_Role") == "Trapper").cast(pl.Int32).sum().alias("count_Trapper"), (pl.col("Operator_Role") == "Intel").cast(pl.Int32).sum().alias("count_Intel"), (pl.col("Operator_Role") == "Support_ATK").cast(pl.Int32).sum().alias("count_Support_ATK"), (pl.col("Operator_Role") == "Demolition").cast(pl.Int32).sum().alias("count_Demolition"), (pl.col("Operator_Role") == "Recon").cast(pl.Int32).sum().alias("count_Recon"), (pl.col("Operator_Role") == "Shield").cast(pl.Int32).sum().alias("count_Shield"), (pl.col("Operator_Role") == "Sniper").cast(pl.Int32).sum().alias("count_Sniper")])
    df_full_enriched = df_full_enriched.join(team_role_stats, on=["matchid", "roundnumber", "team"], how="left")
    opponent_stats = team_role_stats.rename({"team": "opp_team_id", "team_avg_clearance_numeric": "opp_avg_clearance", "team_diversity_score": "opp_diversity_score", "count_Anchor": "opp_count_Anchor", "count_Assault": "opp_count_Assault", "count_Roamer": "opp_count_Roamer", "count_Trapper": "opp_count_Trapper", "count_Intel": "opp_count_Intel", "count_Support_ATK": "opp_count_Support_ATK", "count_Demolition": "opp_count_Demolition", "count_Recon": "opp_count_Recon", "count_Shield": "opp_count_Shield", "count_Sniper": "opp_count_Sniper"})
    df_full_enriched = df_full_enriched.with_columns(pl.when(pl.col("team") == 0).then(pl.lit(1)).otherwise(pl.lit(0)).alias("opp_team_id"))
    df_full_enriched = df_full_enriched.join(opponent_stats, on=["matchid", "roundnumber", "opp_team_id"], how="left").with_columns([(pl.col("team_avg_clearance_numeric") - pl.col("opp_avg_clearance")).alias("Team_Experience_Gap"), (pl.col("My_Team_Score") - pl.col("Opponent_Score")).alias("Match_Pressure_Score")])
    
    all_rounds_ids = df_full_enriched.select(["matchid", "roundnumber"]).unique()
    sampled_ids = all_rounds_ids.sample(n=min(sample_n_rounds, all_rounds_ids.height), seed=SEED)
    df_sample = df_full_enriched.join(sampled_ids, on=["matchid", "roundnumber"]).filter(pl.col("Operator_Role") != "Other")
    
    # Προσθήκη του προθέματος DK_ (Domain Knowledge) στα δικά μας features
    dk_features = {
        "Operator_Role": "DK_Operator_Role", "Match_Phase": "DK_Match_Phase", "Team_Experience_Level": "DK_Team_Experience_Level",
        "Team_Lethal_Score": "DK_Team_Lethal_Score", "Team_Tactical_Score": "DK_Team_Tactical_Score",
        "Team_Experience_Gap": "DK_Team_Experience_Gap", "Match_Pressure_Score": "DK_Match_Pressure_Score",
        "team_diversity_score": "DK_Team_Diversity_Score", "opp_diversity_score": "DK_Opp_Diversity_Score",
        "floor_level": "DK_Floor_Level", "external_soft_walls": "DK_External_Soft_Walls", "crucial_hatches": "DK_Crucial_Hatches",
        "team_shotgun_count": "DK_Team_Shotgun_Count", "team_dmr_count": "DK_Team_DMR_Count",
        "team_shield_count": "DK_Team_Shield_Count", "team_auto_count": "DK_Team_Auto_Count",
        "Prev_Round_Kills_Bucket": "DK_Prev_Round_Kills_Bucket", "Prev_Round_Deaths_Bucket": "DK_Prev_Round_Deaths_Bucket",
        "Match_Kills_Bucket": "DK_Match_Kills_Bucket", "Match_Deaths_Bucket": "DK_Match_Deaths_Bucket",
        "opp_count_Anchor": "DK_Opp_Count_Anchor", "opp_count_Assault": "DK_Opp_Count_Assault",
        "opp_count_Roamer": "DK_Opp_Count_Roamer", "opp_count_Trapper": "DK_Opp_Count_Trapper",
        "opp_count_Intel": "DK_Opp_Count_Intel", "opp_count_Support_ATK": "DK_Opp_Count_Support_ATK",
        "opp_count_Demolition": "DK_Opp_Count_Demolition", "opp_count_Recon": "DK_Opp_Count_Recon",
        "opp_count_Shield": "DK_Opp_Count_Shield", "opp_count_Sniper": "DK_Opp_Count_Sniper"
    }
    
    final_cols_long = [
        "Map_Name", "Skill_Rank", "Role", "Round_Result_Won", "Operator_Name",
        "Clearance_Level", "Primary_Weapon", "Secondary_Gadget", "Operator_Role", "Match_Phase",
        "Prev_Round_Won", "My_Team_Score", "Win_Streak", "Prev_Round_Kills_Bucket",
        "Prev_Round_Deaths_Bucket", "Match_Kills_Bucket", "Match_Deaths_Bucket",
        "Team_Experience_Level", "Team_Lethal_Score", "Team_Tactical_Score", "Team_Experience_Gap",
        "team_diversity_score", "opp_diversity_score", "Match_Pressure_Score", "opp_count_Anchor",
        "opp_count_Assault", "opp_count_Roamer", "opp_count_Trapper", "opp_count_Intel",
        "opp_count_Support_ATK", "opp_count_Demolition", "opp_count_Recon", "opp_count_Shield",
        "opp_count_Sniper", "Opponent_Score", "floor_level", "external_soft_walls", "crucial_hatches",
        "team_shotgun_count", "team_dmr_count", "team_shield_count", "team_auto_count"
    ]

    
    df_sample = df_sample.rename(COLUMN_RENAME_MAP).rename(dk_features).select([dk_features.get(c, c) for c in final_cols_long])
    
    # Μετατροπή της τελικής κλάσης σε κείμενο για το Weka
    df_sample = df_sample.with_columns(pl.col("Round_Result_Won").cast(pl.String))
    
    print(f"Η επεξεργασία ολοκληρώθηκε. Apriori: {df_apriori.shape}, Ρόλοι: {df_apriori_roles.shape}, Δείγμα: {df_sample.shape}")
    return df_apriori, df_apriori_roles, df_sample


### Μετατροπή και Εξαγωγή σε Μορφή ARFF (Weka Integration)

Το περιβάλλον του Weka απαιτεί αυστηρή δήλωση των κατηγορικών (Nominal) μεταβλητών στην επικεφαλίδα (header) του αρχείου `.arff`. Η συνάρτηση `export_to_arff` αυτοματοποιεί αυτή τη μετατροπή, αποτρέποντας την εσφαλμένη αναγνώριση των λογικών μεταβλητών (`true/false`) ή της κλάσης-στόχου (`Round_Result_Won`) ως απλών αριθμητικών (numeric) μεταβλητών από τους αλγορίθμους Ταξινόμησης.


In [ ]:
def export_to_arff(df, relation_name, output_path):
    print(f"📤 Εξαγωγή αρχείου ARFF: {output_path}")
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(f"@RELATION {relation_name}\n\n")
        for col in df.columns:
            dtype_str = str(df[col].dtype).lower()
            if any(t in dtype_str for t in ["int", "float", "decimal"]):
                f.write(f"@ATTRIBUTE \"{col}\" NUMERIC\n")
            elif "bool" in dtype_str:
                f.write(f"@ATTRIBUTE \"{col}\" {{true, false}}\n")
            else:
                vals = df[col].drop_nulls().unique().sort().to_list()
                quoted_vals = [f'"{v}"' for v in vals]
                f.write(f"@ATTRIBUTE \"{col}\" {{{','.join(quoted_vals)}}}\n")
        f.write("\n@DATA\n")
    
    tmp = f"_tmp_{int(time.time())}_{output_path.replace('.arff','')}.csv"
    df.write_csv(tmp, include_header=False, quote_style="always", null_value="?")
    
    # Διόρθωση λέξεων κλειδιών για το αρχείο ARFF
    with open(tmp, "r", encoding="utf-8") as src:
        data = src.read()
        data = data.replace('"?\"', '?')
        data = data.replace('"true"', 'true').replace('"false"', 'false')
        data = data.replace('"True"', 'true').replace('"False"', 'false')
        data = data.replace('True', 'true').replace('False', 'false')
    
    with open(output_path, "a", encoding="utf-8") as dst:
        dst.write(data)
    if os.path.exists(tmp): os.remove(tmp)

if __name__ == "__main__":
    df_apriori, df_apriori_roles, df_sample = run_master_pipelines(sample_n_rounds=40000)
    export_to_arff(df_apriori, "R6_Apriori_Matchups", "1_Weka_Matchups_Apriori.arff")
    export_to_arff(df_sample, "R6_General_Classification", "2_Weka_General_Sample.arff")
    export_to_arff(df_apriori_roles, "R6_Apriori_Roles", "3_Weka_Apriori_Roles.arff")
    print("\n✅ Όλα τα Datasets δημιουργήθηκαν με επιτυχία!")
